# SABR Calibration for Interest Rate Swaptions

This notebook calibrates SABR to swaption implied volatility, validates the fit against real
market quotes, prices a swaption under a bootstrapped discount curve, and computes Greeks.

It runs on one day of Bloomberg VCUB data: USD SOFR swaptions, mid, 2026-07-28, 91 smiles of
nine strikes each, and the matching IRSB SOFR par curve for the same date. No data is generated
anywhere in this notebook. If a slice or a curve tenor needed for pricing is not in the snapshot,
the code raises rather than substituting a fabricated smile or curve.

## The model

$$dF = \alpha \, F^{\beta} \, dW_1, \qquad d\alpha = \nu \, \alpha \, dW_2, \qquad \text{corr}(dW_1, dW_2) = \rho$$

| Parameter | Role |
|-----------|------|
| $\alpha$ | overall vol level, sets ATM |
| $\beta$ | backbone, how vol scales with the rate level. fixed at 0.5 here |
| $\nu$ | vol of vol, width of the smile wings |
| $\rho$ | correlation, direction and steepness of the skew |

$\beta$ is fixed rather than fitted, since calibrating all four parameters at once is unstable:
$\beta$ and $\rho$ both act on the skew and the optimizer cannot tell them apart.

The SABR SDEs have no closed form implied vol. Hagan, Kumar, Lesniewski and Woodward (2002) give an
asymptotic expansion mapping $(\alpha, \beta, \nu, \rho)$ to a Black implied vol for any strike,
fast and differentiable enough for least squares calibration. The VCUB snapshot is quoted in
**normal (Bachelier)** vol, so calibration here uses `hagan_normal_vol`; `hagan_implied_vol`, the
lognormal (Black) version, is used for pricing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import warnings

from sabr_swaption import (
    DARK_THEME,
    hagan_implied_vol, hagan_normal_vol, calibrate_sabr,
    black_price, sabr_price, compute_greeks, price_swaption,
    load_vcub_snapshot, require_slice,
    load_par_curve, bootstrap_discount_curve, discount_factor, curve_max_time,
    annuity_from_curve, forward_swap_rate,
    parse_term,
)

warnings.filterwarnings('ignore')
plt.rcParams.update(DARK_THEME)

## Load VCUB

Raises `FileNotFoundError` if the snapshot file is missing rather than continuing without it.

In [ ]:
vcub_snapshot = load_vcub_snapshot('data/vcub_snapshot.csv')

n_quotes = sum(len(sl['strikes']) for sl in vcub_snapshot.values())
first = next(iter(vcub_snapshot))
sl = vcub_snapshot[first]

print(f"\nslices:         {len(vcub_snapshot)}")
print(f"quotes:         {n_quotes}")
print(f"snapshot date:  {sl['date']}")
print(f"quote kind:     {sl['quote_kind']}")
print(f"vol convention: {sl['vol_model']}")
print(f"\nexample slice {first[0]} into {first[1]}:")
print(f"  atm forward:     {sl['atm_forward']*100:.4f}%")
print(f"  strike offsets:  {np.round(sl['strike_offsets_bp']).astype(int)} (bp)")
print(f"  normal vols:     {np.round(sl['implied_vols']*10000, 2)} (bp)")

## Calibrate

SABR minimizes squared vol error over $(\alpha, \nu, \rho)$ with $\beta$ fixed:

$$\min_{\alpha, \nu, \rho} \sum_{i} \left( \sigma^{\text{mkt}}_i - \sigma^{\text{SABR}}_i(\alpha, \beta, \nu, \rho; K_i, F, T) \right)^2$$

subject to $\alpha > 0$, $\nu > 0$, $|\rho| < 1$. `calibrate_sabr` uses L-BFGS-B with box bounds,
normalizes residuals by the mean market vol so lognormal and normal quotes present the optimizer
with the same scale, and tries a small grid of seeds since a single seed stalls on some expiries.

Every slice in `vcub_snapshot` is calibrated here, in the vol convention it was quoted in.

In [ ]:
calibration_results = {}

print(f"{'expiry':>7} {'tenor':>6} {'alpha':>9} {'nu':>8} {'rho':>8} {'rmse(bp)':>9} {'max(bp)':>8}")
print('-' * 58)

for key, sl in vcub_snapshot.items():
    res = calibrate_sabr(sl['strikes'], sl['atm_forward'], sl['expiry_years'],
                         sl['implied_vols'], beta=0.5, vol_model=sl['vol_model'])
    res['expiry_years'] = sl['expiry_years']
    res['tenor_years'] = sl['tenor_years']
    calibration_results[key] = res
    print(f"{key[0]:>7} {key[1]:>6} {res['alpha']:>9.6f} {res['nu']:>8.4f} "
          f"{res['rho']:>8.4f} {res['rmse']*10000:>9.3f} {res['max_err']*10000:>8.3f}")

## Validate

Real dealer quotes at nine strikes went into the fit above, and nothing about them was generated
by this code. RMSE and max error against those quotes is therefore a genuine market fit test, not
a self consistency check. A slice passes at RMSE under 5bp.

In [ ]:
VALIDATION_RMSE_LIMIT_BP = 5.0

rmses_bp = np.array([r['rmse'] for r in calibration_results.values()]) * 10000
worst_key = max(calibration_results, key=lambda k: calibration_results[k]['rmse'])
n_over = int((rmses_bp >= VALIDATION_RMSE_LIMIT_BP).sum())
passed = n_over == 0

banner = '\033[42m\033[30m' if passed else '\033[41m\033[37m'
reset = '\033[0m'
print(banner + ' ' * 78 + reset)
if passed:
    print(banner + '  ✅ code validated against real VCUB data'.ljust(77) + reset)
else:
    print(banner + f'  FLAGGED  {n_over} slice(s) at or above the {VALIDATION_RMSE_LIMIT_BP:.0f}bp limit'.ljust(78) + reset)
print(banner + f"  {len(calibration_results)} VCUB slices, max RMSE {rmses_bp.max():.3f}bp "
               f"({worst_key[0]} into {worst_key[1]}), mean {rmses_bp.mean():.3f}bp".ljust(78) + reset)
print(banner + '  scope: SABR fit to raw VCUB strike quotes, a genuine market fit test.'.ljust(78) + reset)
print(banner + ' ' * 78 + reset)

In [ ]:
PLOT_SLICES = [('1Mo', '1Yr'), ('1Yr', '5Yr'), ('5Yr', '10Yr'),
              ('10Yr', '10Yr'), ('20Yr', '10Yr'), ('30Yr', '30Yr')]

fig, axes = plt.subplots(2, 3, figsize=(18, 8), squeeze=False)
for ax, key in zip(axes.flat, PLOT_SLICES):
    sl = vcub_snapshot[key]
    res = calibration_results[key]
    vol_fn = hagan_normal_vol if sl['vol_model'] == 'normal' else hagan_implied_vol
    scale = 10000 if sl['vol_model'] == 'normal' else 100
    unit = 'bp' if sl['vol_model'] == 'normal' else '%'

    fine = np.linspace(sl['strikes'][0], sl['strikes'][-1], 200)
    fit = vol_fn(fine, sl['atm_forward'], sl['expiry_years'],
                res['alpha'], res['beta'], res['nu'], res['rho'])

    ax.plot(sl['strike_offsets_bp'], sl['implied_vols'] * scale, 'o',
            color='#00d2ff', markersize=8, label='VCUB', zorder=5)
    ax.plot((fine - sl['atm_forward']) * 10000, fit * scale, '-',
            color='#4ade80', linewidth=2.5, label='SABR fit')
    ax.axvline(0, color='#888', linestyle=':', alpha=0.4)
    ax.set_title(f"{key[0]} into {key[1]}  (RMSE {res['rmse']*10000:.2f}bp)",
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('strike offset (bp)')
    ax.set_ylabel(f'implied vol ({unit})')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

date = next(iter(vcub_snapshot.values()))['date']
plt.suptitle(f'VCUB market quotes vs SABR fit, snapshot {date}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

A single day snapshot validates the method on one cross section. It does not establish a time
series, it says nothing about parameter stability day to day, and because each slice is fitted
independently the resulting surface is not arbitrage free across expiries. Fitting well at every
slice and being arbitrage free across the grid are different properties, and only the first is
tested here.

## SABR vs market at a strike

Pick a slice and a strike. Reports the SABR vol at that strike, the nearest quoted vol, and the
difference. Looking up a slice not in the snapshot raises `KeyError` rather than substituting one.

In [ ]:
# sabr vol at a strike against the nearest quoted vol for that slice
def compare_at_strike(key, strike):
    sl = require_slice(vcub_snapshot, *key)
    res = calibration_results[key]
    vol_fn = hagan_normal_vol if res['vol_model'] == 'normal' else hagan_implied_vol
    unit = 'bp' if res['vol_model'] == 'normal' else '%'
    scale = 10000 if res['vol_model'] == 'normal' else 100

    sabr_vol = float(vol_fn(strike, sl['atm_forward'], res['expiry_years'],
                            res['alpha'], res['beta'], res['nu'], res['rho']))
    idx = int(np.argmin(np.abs(sl['strikes'] - strike)))
    ref_vol = float(sl['implied_vols'][idx])

    print('SABR vs real market (VCUB)')
    print(f"  slice          {key[0]} into {key[1]}   {sl['quote_kind']}")
    print(f"  atm forward    {sl['atm_forward']*100:.4f}%")
    print(f"  strike         {strike*100:.4f}%  ({(strike - sl['atm_forward'])*10000:+.1f}bp from ATM)")
    print(f"  SABR vol       {sabr_vol*scale:.3f}{unit}  ({res['vol_model']} convention)")
    print(f"  nearest quote  {ref_vol*scale:.3f}{unit}  at strike {sl['strikes'][idx]*100:.4f}%")
    print(f"  difference     {(sabr_vol - ref_vol)*scale:+.3f}{unit}")
    return {'sabr_vol': sabr_vol, 'ref_vol': ref_vol}


COMPARE_KEY = ('5Yr', '10Yr')
compare_strike = vcub_snapshot[COMPARE_KEY]['atm_forward'] + 0.0050
compare_at_strike(COMPARE_KEY, compare_strike)

## Pricing

The cells above price one unit of annuity. A real number needs a real annuity, so the discount
curve is bootstrapped from the SOFR par curve of the same snapshot date: pillars inside the first
coupon date are simple compounded deposits, the rest bootstrap on a semi-annual grid, and discount
factors interpolate log-linearly between pillars. The annuity is the sum of Actual/360 day count
weighted discount factors over the fixed leg, the forward swap rate is derived from the curve
rather than assumed, and Black-76 runs under that annuity measure.

The curve's last pillar is 30Y, but some slices need discount factors out to 60Y (a 30Y expiry
into a 30Y swap). Past the last pillar the code holds the instantaneous forward flat, capped at
twice the curve's own reach, which covers every slice in this snapshot; a curve gap beyond that
cap raises rather than extrapolating further. Slices relying on that extrapolation are flagged
`extrap` in the summary table below. `load_par_curve` raises `FileNotFoundError` if the curve file
is missing, the same as the VCUB loader.

In [ ]:
par_rates = load_par_curve('data/sofr_curve.csv')
curve = bootstrap_discount_curve(par_rates)

times, dfs = curve
print(f"\nbootstrapped {len(times)-1} pillars out to {times[-1]:.1f}Y")
print(f"  df(1Y) = {float(discount_factor(curve, 1)):.4f}   "
      f"df(10Y) = {float(discount_factor(curve, 10)):.4f}   "
      f"df(30Y) = {float(discount_factor(curve, 30)):.4f}")

print('\nforward cross check, curve against the snapshot quotes:')
for k in [('1Yr', '10Yr'), ('5Yr', '10Yr'), ('10Yr', '10Yr'), ('30Yr', '30Yr')]:
    sl_k = vcub_snapshot[k]
    f_curve = forward_swap_rate(curve, sl_k['expiry_years'], sl_k['tenor_years'])
    tail = sl_k['expiry_years'] + sl_k['tenor_years']
    note = '  (extrapolated past the last curve pillar)' if tail > curve_max_time(curve) else ''
    print(f"  {k[0]:>5} into {k[1]:<5} curve {f_curve*100:.4f}%   "
          f"snapshot {sl_k['atm_forward']*100:.4f}%   "
          f"gap {(f_curve - sl_k['atm_forward'])*10000:+6.1f}bp{note}")

In [ ]:
PRICE_KEY = ('5Yr', '10Yr')
sl = require_slice(vcub_snapshot, *PRICE_KEY)
res = calibration_results[PRICE_KEY]
notional = 100_000_000

expiry_y, tenor_y = res['expiry_years'], res['tenor_years']
strike = sl['atm_forward'] + 0.0050

fwd_curve = forward_swap_rate(curve, expiry_y, tenor_y)
ann = annuity_from_curve(curve, expiry_y, tenor_y)

quote = price_swaption(sl['atm_forward'], strike, expiry_y, tenor_y,
                       res['alpha'], res['beta'], res['nu'], res['rho'],
                       curve=curve, notional=notional)
greeks = compute_greeks(sl['atm_forward'], strike, expiry_y, res['alpha'],
                        res['beta'], res['nu'], res['rho'])

print(f"  instrument      {PRICE_KEY[0]} payer swaption into {PRICE_KEY[1]} swap")
print(f"  snapshot date   {sl['date']}")
print(f"  vol quote kind  {sl['quote_kind']}")
print(f"  annuity         {ann:.4f}   [bootstrapped SOFR curve, ACT/360 semi-annual]")
print(f"  fwd from curve  {fwd_curve*100:.4f}%")
print(f"  fwd from vols   {sl['atm_forward']*100:.4f}%  (snapshot ATM forward)")
print(f"  strike          {strike*100:.4f}%  (ATM + 50bp)")
print(f"  SABR lognormal vol  {quote['vol']*100:.2f}%")
print(f"  price (unit annuity) {quote['unit_price']:.6f}")
print(f"  price               ${quote['price']:,.0f}  on ${notional:,.0f} notional")
print(f"  vega per 1bp        ${greeks['vega'] * ann * notional:,.0f}")
print()
print(f"  vols and curve are both from {sl['date']}, priced on real VCUB data throughout.")
print(f"  curve and snapshot forwards agree to {(fwd_curve - sl['atm_forward'])*10000:+.1f}bp,")
print('  which is the bootstrap cross check above applied to this slice.')

## Greeks

Vega, gamma and vanna by finite difference on the calibrated SABR parameters: Hagan maps
parameters to an implied vol, Black-76 maps that vol to a price, and `compute_greeks` bumps the
inputs and differences the prices. Vega is per 1bp move in $\alpha$, gamma is the second
derivative in the forward, and vanna is $\partial^2 P / \partial F \partial \sigma$, which tells a
desk how its vega exposure shifts as rates move.

In [ ]:
greeks_by_strike = [compute_greeks(sl['atm_forward'], K, expiry_y,
                                   res['alpha'], res['beta'], res['nu'], res['rho'])
                    for K in sl['strikes']]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
panels = [('price', 'price', '#00d2ff'), ('vega', 'vega (per 1bp)', '#ff6b6b'),
          ('gamma', 'gamma', '#ffd93d'), ('vanna', 'vanna', '#a855f7')]

for ax, (key_, label, color) in zip(axes.flat, panels):
    ax.plot(sl['strike_offsets_bp'], [g[key_] for g in greeks_by_strike],
            'o-', color=color, linewidth=2, markersize=7)
    ax.axvline(0, color='#888', linestyle=':', alpha=0.4)
    ax.set_xlabel('strike offset (bp)')
    ax.set_ylabel(label)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle(f"SABR Greeks, {PRICE_KEY[0]} into {PRICE_KEY[1]} payer, VCUB {sl['date']}",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Vega and gamma both peak near ATM. Vanna changes sign there, which is why a book with skew
exposure has to be watched on both axes at once.

## Summary

One row per VCUB slice: source, calibrated parameters, fit error, price. `extrap` flags slices
whose last payment falls beyond the curve's final pillar, where the annuity relies on flat forward
extrapolation.

In [ ]:
rows = []
for key, res in calibration_results.items():
    sl = vcub_snapshot[key]
    priced = price_swaption(sl['atm_forward'], sl['atm_forward'],
                            res['expiry_years'], res['tenor_years'],
                            res['alpha'], res['beta'], res['nu'], res['rho'], curve=curve)
    rows.append({
        'expiry': key[0], 'tenor': key[1],
        'alpha': res['alpha'], 'nu': res['nu'], 'rho': res['rho'],
        'rmse_bp': res['rmse'] * 10000, 'price': priced['unit_price'],
        'flag': 'extrap' if res['expiry_years'] + res['tenor_years'] > curve_max_time(curve) else '',
        'order': (res['expiry_years'], res['tenor_years']),
    })

rows.sort(key=lambda r: r['order'])

header = (f"{'expiry':>7} {'tenor':>6} {'alpha':>9} {'nu':>8} {'rho':>8} "
          f"{'rmse(bp)':>9} {'price':>10} {'flag':>8}")
print(header)
print('-' * len(header))
for r in rows:
    print(f"{r['expiry']:>7} {r['tenor']:>6} {r['alpha']:>9.6f} "
          f"{r['nu']:>8.4f} {r['rho']:>8.4f} {r['rmse_bp']:>9.3f} {r['price']:>10.6f} "
          f"{r['flag']:>8}")

print()
print(f"all {len(rows)} rows are VCUB, snapshot date {sl['date']}. price is ATM, per unit notional,")
print('under the bootstrapped SOFR annuity.')

## Vol surface in three dimensions

The VCUB snapshot across expiries at a fixed swap tenor, fitted slice by slice and interpolated
across the expiry axis. Cyan points are the quoted vols, the mesh is the SABR fit.

In [ ]:
from scipy.interpolate import RegularGridInterpolator

SURFACE_TENOR = '10Yr'
surface_keys = sorted((k for k in vcub_snapshot if k[1] == SURFACE_TENOR),
                      key=lambda k: parse_term(k[0]))

k_min = max(vcub_snapshot[k]['strikes'].min() for k in surface_keys)
k_max = min(vcub_snapshot[k]['strikes'].max() for k in surface_keys)
grid_strikes = np.linspace(k_min, k_max, 80)

surface_exp = [parse_term(k[0]) for k in surface_keys]
vol_matrix = np.zeros((len(surface_keys), len(grid_strikes)))
for i, key in enumerate(surface_keys):
    sl_i, res_i = vcub_snapshot[key], calibration_results[key]
    vol_matrix[i, :] = hagan_normal_vol(grid_strikes, sl_i['atm_forward'], res_i['expiry_years'],
                                        res_i['alpha'], res_i['beta'], res_i['nu'], res_i['rho']) * 10000

fine_exp = np.linspace(min(surface_exp), max(surface_exp), 60)
interp = RegularGridInterpolator((np.array(surface_exp), grid_strikes), vol_matrix,
                                 method='linear', bounds_error=False, fill_value=None)
strike_mesh, expiry_mesh = np.meshgrid(grid_strikes, fine_exp)
vol_smooth = interp(np.column_stack([expiry_mesh.ravel(), strike_mesh.ravel()])).reshape(expiry_mesh.shape)

fig = plt.figure(figsize=(14, 9))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(strike_mesh * 100, expiry_mesh, vol_smooth,
                       cmap='plasma', edgecolor='none', alpha=0.92)

for key in surface_keys:
    sl_i = vcub_snapshot[key]
    ax.scatter(sl_i['strikes'] * 100, np.full_like(sl_i['strikes'], parse_term(key[0])),
               sl_i['implied_vols'] * 10000, color='#00d2ff', s=25,
               edgecolors='white', linewidth=0.5, zorder=10)

ax.set_xlabel('strike (%)', labelpad=10)
ax.set_ylabel('option expiry (years)', labelpad=10)
ax.set_zlabel('normal vol (bp)', labelpad=10)
date = vcub_snapshot[surface_keys[0]]['date']
ax.set_title(f'Swaption normal vol surface, {SURFACE_TENOR} tenor, VCUB {date}',
             fontsize=13, fontweight='bold', pad=20)
ax.view_init(elev=25, azim=-55)
for pane in (ax.xaxis, ax.yaxis, ax.zaxis):
    pane.pane.fill = False
fig.colorbar(surf, ax=ax, shrink=0.55, aspect=12, pad=0.1, label='normal vol (bp)')
plt.tight_layout()
plt.show()

## Limitations

Each expiry and tenor is calibrated independently, so fitting every slice to under 5bp does not
make the surface arbitrage free across expiries, and nothing here checks calendar or butterfly
conditions on the fitted grid. There is no shift term, so the model is unusable at or below zero
rates. Hagan's expansion degrades at very long expiries and far strikes, where Obloj's correction
or a direct normal SABR parameterisation does better. Discounting uses a single SOFR curve with no
basis or multi-curve adjustment, and swaps running past its last pillar at 30Y rely on flat forward
extrapolation, so those annuities are indicative rather than desk numbers. One day of data
validates the method and says nothing about parameter stability over time.